# Data prep — step by step

This notebook walks through the same pipeline as **`scripts/build_datasets.py`**.

| Steps | What | Code location |
|-------|------|---------------|
| 1–5 | Superstore spine, context, CRM, dim, Telco/Instacart/Online Retail | Inline below |
| 6–8 | Feature store, uplift, NBA | `scripts/feature_store.py`, `scripts/modeling_extensions.py` |
| 9 | Save all Parquet files to `data/modeling/` | Mirrors `build_datasets.run()` |

Full reference: `DATA_PIPELINE.md` · Modeling guide: `MODELING.md`

In [1]:
import io
import sys
import time
from pathlib import Path

import holidays
import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", None)


def find_project_root() -> Path:
    """Resolve repo root whether the kernel cwd is / or /Notebooks."""
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "scripts" / "build_datasets.py").exists():
            return candidate
    return path


PROJECT_ROOT = find_project_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_MODELING = PROJECT_ROOT / "data" / "modeling"

# Shared modules used by build_datasets.py (Steps 6–8)
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from feature_store import build_customer_features
from modeling_extensions import (
    build_instacart_baskets,
    build_instacart_orders,
    build_nba_catalog,
    build_nba_events,
    build_online_retail_customers,
    build_uplift_campaigns,
    load_online_retail,
)

US_SUPERSTORE_URL = (
    "https://raw.githubusercontent.com/ThigasSantos/BaseSuperStore/main/sales.csv"
)
US_TELCO_URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/"
    "Telco-Customer-Churn.csv"
)
CONTEXT_MONTH_START = "2016-01"
CONTEXT_MONTH_END = "2019-12"
CONTEXT_DATE_START = "2016-01-01"
CONTEXT_DATE_END = "2019-12-31"
MACRO_FETCH_MONTH_START = "2015-01"
MACRO_FETCH_DATE_START = "2015-01-01"

US_MACRO_URL = (
    "https://fred.stlouisfed.org/graph/fredgraph.csv?"
    "id=CPIAUCSL,UNRATE,FEDFUNDS"
    f"&cosd={MACRO_FETCH_DATE_START}&coed={CONTEXT_DATE_END}"
)
MACRO_COLUMNS = [
    "month",
    "cpi_index",
    "inflation_mom",
    "inflation_yoy",
    "unemployment_rate",
    "interest_rate",
]

WEATHER_DAILY_VARS = (
    "temperature_2m_mean,rain_sum,snowfall_sum,precipitation_sum,"
    "weather_code,wind_gusts_10m_max"
)
STORM_WEATHER_CODES = {65, 67, 75, 77, 82, 85, 86, 95, 96, 99}
MAJOR_STORM_WIND_KMH = 75.0
# Open-Meteo free tier: 1 concurrent request/IP. Fetch order dates only (not every calendar day).
OPEN_METEO_REQUEST_DELAY_SEC = 10
WEATHER_DATE_CHUNK = 100


class OpenMeteoQuotaExceeded(Exception):
    pass

## Step 1 — Download & clean Superstore (base fact table)

**Source:** public CSV on GitHub  
**Grain:** one row = one **order line item** (same `order_id` can repeat)

In [3]:
superstore_path = DATA_RAW / "us_superstore" / "superstore_sales.csv"

if not superstore_path.exists():
    superstore_path.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading Superstore...")
    r = requests.get(US_SUPERSTORE_URL, timeout=120)
    r.raise_for_status()
    raw = pd.read_csv(io.StringIO(r.text))
    print("Raw columns:", list(raw.columns))
    print("Raw shape:", raw.shape)

    superstore = raw.rename(columns={
        "OrderID": "order_id", "OrderDate": "order_date", "ShipDate": "ship_date",
        "ShipMode": "ship_mode", "CustomerID": "customer_id", "CustomerName": "customer_name",
        "Segment": "segment", "Country": "country", "City": "city", "State": "state",
        "Postal Code": "postal_code", "Region": "region", "ProductID": "product_id",
        "Category": "category", "Sub-Category": "sub_category", "ProductName": "product_name",
        "Sales": "sales", "Quantity": "quantity", "Discount": "discount", "Profit": "profit",
    })
    superstore["order_date"] = pd.to_datetime(superstore["order_date"], dayfirst=True)
    superstore["ship_date"] = pd.to_datetime(superstore["ship_date"], dayfirst=True)
    superstore = superstore[superstore["country"] == "United States"].copy()
    superstore.to_csv(superstore_path, index=False)
    print(f"Saved cleaned US-only data → {superstore_path}")
else:
    print(f"Using cached file → {superstore_path}")
    superstore = pd.read_csv(superstore_path, parse_dates=["order_date", "ship_date"])

print("Shape:", superstore.shape)
print("Regions:", superstore["region"].unique())
superstore.head(3)

Using cached file → c:\Users\ahmed\Desktop\Projects\LoyaltySim-AI\data\raw\us_superstore\superstore_sales.csv
Shape: (9994, 22)
Regions: <StringArray>
['South', 'West', 'Central', 'East']
Length: 4, dtype: str


,order_id,order_date,OrderYear,Order Quarter,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,CA-2018-152156,2018-11-08,2018,3,2018-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,CA-2018-152156,2018-11-08,2018,3,2018-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,CA-2018-138688,2018-06-12,2018,4,2018-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


In [ ]:
# Grain check: many line items per order, many orders per customer
print("Line items (rows):", len(superstore))
print("Unique orders:", superstore["order_id"].nunique())
print("Unique customers:", superstore["customer_id"].nunique())

In [ ]:
print(superstore["order_date"].dt.strftime("%Y-%m").min())
print(superstore["order_date"].dt.strftime("%Y-%m").max())

## Step 2a — Download macro context (CPI, unemployment, interest rate)

**Source:** FRED monthly (`CPIAUCSL`, `UNRATE`, `FEDFUNDS`)  
**Grain:** one row = one **month** (`YYYY-MM`)  
**Note:** Fetches from 2015 so YoY inflation is defined for 2016+; output is trimmed to Superstore range.

In [2]:
macro_path = DATA_RAW / "external" / "us_macro_monthly.csv"

needs_refresh = True
if macro_path.exists():
    cached = pd.read_csv(macro_path)
    needs_refresh = (
        cached.empty
        or not set(MACRO_COLUMNS).issubset(cached.columns)
        or cached["month"].min() > CONTEXT_MONTH_START
        or cached["month"].max() < CONTEXT_MONTH_END
    )

if not needs_refresh:
    print(f"Using cached file → {macro_path}")
    macro = cached
else:
    print("Downloading macro series from FRED...")
    try:
        r = requests.get(US_MACRO_URL, timeout=90)
        r.raise_for_status()
        raw = pd.read_csv(io.StringIO(r.text))
        macro = raw.rename(columns={
            "observation_date": "month",
            "CPIAUCSL": "cpi_index",
            "UNRATE": "unemployment_rate",
            "FEDFUNDS": "interest_rate",
        })
        macro["month"] = pd.to_datetime(macro["month"]).dt.to_period("M").astype(str)
        for col in ("cpi_index", "unemployment_rate", "interest_rate"):
            macro[col] = pd.to_numeric(macro[col], errors="coerce")
        macro = macro.dropna(subset=["cpi_index"])
        macro["inflation_mom"] = macro["cpi_index"].pct_change().round(5)
        if macro["month"].min() <= MACRO_FETCH_MONTH_START:
            macro["inflation_yoy"] = macro["cpi_index"].pct_change(12).round(5)
    except (requests.RequestException, ValueError, KeyError):
        print("FRED unreachable — using flat CPI placeholder")
        months = pd.period_range(CONTEXT_MONTH_START, CONTEXT_MONTH_END, freq="M").astype(str)
        macro = pd.DataFrame({
            "month": months,
            "cpi_index": 240.0,
            "unemployment_rate": np.nan,
            "interest_rate": np.nan,
        })
        macro["inflation_mom"] = macro["cpi_index"].pct_change().round(5)

    macro = macro[
        (macro["month"] >= CONTEXT_MONTH_START) & (macro["month"] <= CONTEXT_MONTH_END)
    ].copy()
    macro_path.parent.mkdir(parents=True, exist_ok=True)
    macro[[c for c in MACRO_COLUMNS if c in macro.columns]].to_csv(macro_path, index=False)
    print(f"Saved → {macro_path}")

macro = macro[[c for c in MACRO_COLUMNS if c in macro.columns]].copy()
macro.head()

Using cached file → c:\Users\ahmed\Desktop\Projects\LoyaltySim-AI\data\raw\external\us_macro_monthly.csv


,month,cpi_index,inflation_mom,inflation_yoy,unemployment_rate,interest_rate
0,2016-01,237.652,-0.00046,0.01238,4.8,0.34
1,2016-02,237.336,-0.00133,0.00847,4.9,0.38
2,2016-03,238.080,0.00313,0.00892,5.0,0.36
3,2016-04,238.992,0.00383,0.01173,5.1,0.37
4,2016-05,239.557,0.00236,0.01078,4.8,0.37


In [ ]:
print(macro['month'].min())
print(macro['month'].max())

## Step 2b — Download weather by city + order date

**Source:** Open-Meteo geocoding + archive API  
**Grain:** one row = one **date + city + state** that appears in Superstore orders (not every calendar day)  
**Range:** order dates in 2016-01 → 2019-12 (~4,735 rows, not 604 cities × 1,461 days)  
**Fields:** temp, rain, snow, precipitation, wind gusts, WMO weather code, storm flags

In [5]:
def geocode_city(city: str, state: str) -> dict | None:
    r = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 10, "country": "US"},
        timeout=30,
    )
    r.raise_for_status()
    results = r.json().get("results") or []
    match = next(
        (x for x in results if x.get("admin1", "").lower() == state.lower()),
        None,
    )
    if match is None and results:
        match = results[0]
    if match is None:
        return None
    return {
        "city": city,
        "state": state,
        "latitude": match["latitude"],
        "longitude": match["longitude"],
        "timezone": match["timezone"],
    }


def add_weather_flags(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.rename(columns={
        "time": "date",
        "temperature_2m_mean": "temp_c",
        "rain_sum": "rain_mm",
        "snowfall_sum": "snow_cm",
        "precipitation_sum": "precip_mm",
        "wind_gusts_10m_max": "wind_gust_kmh",
    })
    out["had_rain"] = out["rain_mm"].fillna(0) > 0
    out["had_snow"] = out["snow_cm"].fillna(0) > 0
    out["had_major_storm"] = (
        out["weather_code"].isin(STORM_WEATHER_CODES)
        | (out["wind_gust_kmh"].fillna(0) >= MAJOR_STORM_WIND_KMH)
    )
    return out


order_keys = (
    superstore.assign(date=superstore["order_date"].dt.normalize())
    .loc[
        superstore["order_date"].between(CONTEXT_DATE_START, CONTEXT_DATE_END),
        ["date", "city", "state"],
    ]
    .drop_duplicates()
)
cities = order_keys[["city", "state"]].drop_duplicates()
dates_by_city = (
    order_keys.groupby(["city", "state"])["date"]
    .apply(lambda s: sorted(pd.to_datetime(s).dt.strftime("%Y-%m-%d").unique()))
    .to_dict()
)
coords_path = DATA_RAW / "external" / "city_coordinates.csv"
weather_path = DATA_RAW / "external" / "us_weather_by_city.csv"

print(f"Need weather for {len(order_keys):,} order-date+city rows across {len(cities)} cities")

if coords_path.exists() and len(pd.read_csv(coords_path)) >= len(cities):
    print(f"Using cached coordinates → {coords_path}")
    coords = pd.read_csv(coords_path)
else:
    print(f"Geocoding {len(cities)} Superstore cities...")
    coord_rows = []
    for city, state in cities.itertuples(index=False):
        row = geocode_city(city, state)
        if row is not None:
            coord_rows.append(row)
    coords = pd.DataFrame(coord_rows)
    coords_path.parent.mkdir(parents=True, exist_ok=True)
    coords.to_csv(coords_path, index=False)
    print(f"Saved → {coords_path}")

def city_weather_complete(cached: pd.DataFrame, city: str, state: str) -> bool:
    needed = order_keys.loc[
        (order_keys["city"] == city) & (order_keys["state"] == state), "date"
    ].dt.normalize()
    if needed.empty:
        return True
    have = cached.loc[
        (cached["city"] == city) & (cached["state"] == state), "date"
    ].dt.normalize()
    return set(needed) <= set(have)


def open_meteo_429_reason(response):
    try:
        return str(response.json().get("reason", ""))
    except ValueError:
        return response.text[:200]


def open_meteo_daily_frame(payload):
    # Comma-separated dates return one response object per date (a list).
    if isinstance(payload, list):
        parts = [pd.DataFrame(item["daily"]) for item in payload]
        return pd.concat(parts, ignore_index=True)
    return pd.DataFrame(payload["daily"])


def fetch_weather_archive(latitude, longitude, timezone, dates):
    frames = []
    for i in range(0, len(dates), WEATHER_DATE_CHUNK):
        chunk = dates[i : i + WEATHER_DATE_CHUNK]
        date_str = ",".join(chunk)
        params = {
            "latitude": latitude,
            "longitude": longitude,
            "start_date": date_str,
            "end_date": date_str,
            "daily": WEATHER_DAILY_VARS,
            "timezone": timezone,
        }
        attempt = 0
        while True:
            r = requests.get(
                "https://archive-api.open-meteo.com/v1/archive",
                params=params,
                timeout=120,
            )
            if r.status_code == 429:
                reason = open_meteo_429_reason(r)
                if "daily" in reason.lower():
                    raise OpenMeteoQuotaExceeded(
                        reason or "Daily API request limit exceeded. Re-run tomorrow."
                    )
                wait = int(r.headers.get("Retry-After", 0)) or min(60 * (2 ** attempt), 600)
                attempt += 1
                if attempt >= 12:
                    raise OpenMeteoQuotaExceeded(
                        reason or f"Rate limited after {attempt} retries."
                    )
                print(f"  rate limited — waiting {wait}s (attempt {attempt}/12)")
                time.sleep(wait)
                continue
            r.raise_for_status()
            frames.append(open_meteo_daily_frame(r.json()))
            break
        if i + WEATHER_DATE_CHUNK < len(dates):
            time.sleep(OPEN_METEO_REQUEST_DELAY_SEC)
    return pd.concat(frames, ignore_index=True)


cached_weather = pd.DataFrame()
if weather_path.exists():
    cached_weather = pd.read_csv(weather_path, parse_dates=["date"])
    trimmed = cached_weather.merge(order_keys, on=["date", "city", "state"], how="inner")
    if len(trimmed) < len(cached_weather):
        print(f"Trimmed cache {len(cached_weather):,} → {len(trimmed):,} rows (order dates only)")
        trimmed.to_csv(weather_path, index=False)
    cached_weather = trimmed

pending = [
    row
    for row in coords.itertuples(index=False)
    if not city_weather_complete(cached_weather, row.city, row.state)
]

if not pending:
    print(f"Using cached file → {weather_path}")
    weather = cached_weather
else:
    done = len(coords) - len(pending)
    if done:
        print(f"Resuming weather download ({done}/{len(coords)} cities already cached)")
    print(
        f"Downloading weather for {len(pending)} cities "
        f"(order dates only, ~{OPEN_METEO_REQUEST_DELAY_SEC}s between calls)..."
    )
    weather_path.parent.mkdir(parents=True, exist_ok=True)
    quota_hit = False
    try:
        for i, row in enumerate(pending, start=1):
            dates = dates_by_city[(row.city, row.state)]
            frame = add_weather_flags(
                fetch_weather_archive(row.latitude, row.longitude, row.timezone, dates)
            )
            frame["city"] = row.city
            frame["state"] = row.state
            cached_weather = pd.concat([cached_weather, frame], ignore_index=True)
            cached_weather["date"] = pd.to_datetime(cached_weather["date"])
            cached_weather = cached_weather.merge(order_keys, on=["date", "city", "state"], how="inner")
            cached_weather.to_csv(weather_path, index=False)
            if i % 25 == 0 or i == len(pending):
                print(f"  {done + i}/{len(coords)} cities | {len(cached_weather):,} rows saved")
            if i < len(pending):
                time.sleep(OPEN_METEO_REQUEST_DELAY_SEC)
    except OpenMeteoQuotaExceeded as exc:
        quota_hit = True
        print(f"Weather download paused: {exc}")
        print("Progress saved — re-run this cell later to resume.")

    weather = cached_weather
    if not quota_hit:
        print(f"Saved → {weather_path}")

print(
    "Weather range:",
    weather["date"].min().date(),
    "→",
    weather["date"].max().date(),
    "| cities:",
    weather[["city", "state"]].drop_duplicates().shape[0],
)
weather.head()

Need weather for 4,735 order-date+city rows across 604 cities
Using cached coordinates → c:\Users\ahmed\Desktop\Projects\LoyaltySim-AI\data\raw\external\city_coordinates.csv
Resuming weather download (153/604 cities already cached)
  178/604 cities | 3,266 rows saved
  203/604 cities | 3,402 rows saved
  228/604 cities | 3,535 rows saved
  253/604 cities | 3,661 rows saved
  278/604 cities | 3,804 rows saved
  303/604 cities | 3,948 rows saved
  328/604 cities | 4,063 rows saved
  353/604 cities | 4,167 rows saved
  378/604 cities | 4,258 rows saved
  403/604 cities | 4,338 rows saved
  428/604 cities | 4,392 rows saved
  453/604 cities | 4,459 rows saved
  478/604 cities | 4,512 rows saved
  503/604 cities | 4,575 rows saved
  528/604 cities | 4,621 rows saved
  553/604 cities | 4,662 rows saved
  578/604 cities | 4,705 rows saved
  603/604 cities | 4,734 rows saved
  604/604 cities | 4,735 rows saved
Saved → c:\Users\ahmed\Desktop\Projects\LoyaltySim-AI\data\raw\external\us_weather_b

,date,temp_c,rain_mm,snow_cm,precip_mm,weather_code,wind_gust_kmh,had_rain,had_snow,had_major_storm,city,state
0,2016-01-06,0.3,0.0,0.0,0.0,3,28.1,False,False,False,Henderson,Kentucky
1,2016-04-21,17.8,14.0,0.0,14.0,63,40.0,True,False,False,Henderson,Kentucky
2,2016-05-09,19.9,2.5,0.0,2.5,53,42.8,True,False,False,Henderson,Kentucky
3,2016-12-13,1.9,0.0,0.0,0.0,3,33.5,False,False,False,Henderson,Kentucky
4,2017-01-09,-0.4,0.0,0.0,0.0,3,45.0,False,False,False,Henderson,Kentucky


## Step 2c — Join context onto Superstore

Three enrichments, applied in order:

1. **Calendar features** — computed on each row (no join)
2. **CPI** — `LEFT JOIN` on `month`
3. **Weather** — `LEFT JOIN` on `date` + `city` + `state`

In [8]:
df = superstore.copy()

# --- 1) Calendar features (no join) ---
from datetime import date, timedelta

us_holidays = holidays.country_holidays("US", years=range(2015, 2020))

# Retail spending days beyond federal holidays:
# Valentine's Day, Mother's Day, Super Bowl Sunday, Black Friday, Christmas Eve
retail_spending_days = set()
for year in range(2015, 2020):
    retail_spending_days.add(date(year, 2, 14))
    retail_spending_days.add(date(year, 12, 24))
    may1 = date(year, 5, 1)
    retail_spending_days.add(date(year, 5, 1 + (6 - may1.weekday()) % 7 + 7))  # 2nd Sunday in May
    nov1 = date(year, 11, 1)
    thanksgiving = date(year, 11, 1 + (3 - nov1.weekday()) % 7 + 21)  # 4th Thursday in Nov
    retail_spending_days.add(thanksgiving + timedelta(days=1))  # Black Friday
    feb1 = date(year, 2, 1)
    retail_spending_days.add(feb1 + timedelta(days=(6 - feb1.weekday()) % 7))  # Super Bowl Sunday

df["date"] = df["order_date"].dt.normalize()
df["month"] = df["order_date"].dt.to_period("M").astype(str)
df["is_us_federal_holiday"] = df["date"].dt.date.map(lambda d: d in us_holidays)
df["is_retail_spending_day"] = df["date"].dt.date.isin(retail_spending_days)
df["day_of_week"] = df["order_date"].dt.dayofweek
df["is_friday"] = df["day_of_week"] == 4
df["is_month_start"] = df["order_date"].dt.day <= 5
df["is_month_end"] = df["order_date"].dt.day >= 25

print("After calendar features:", df.shape)
df[[
    "order_date", "month", "is_us_federal_holiday", "is_retail_spending_day",
    "day_of_week", "is_friday", "is_month_start", "is_month_end",
]].head(3)

After calendar features: (9994, 30)


,order_date,month,is_us_federal_holiday,is_retail_spending_day,day_of_week,is_friday,is_month_start,is_month_end
0,2018-11-08,2018-11,False,False,3,False,False,False
1,2018-11-08,2018-11,False,False,3,False,False,False
2,2018-06-12,2018-06,False,False,1,False,False,False


In [9]:
# --- 2) JOIN macro (CPI) on month ---
print("Join keys: superstore.month = macro.month")
print("Macro rows:", len(macro), "| Unique months in superstore:", df["month"].nunique())

df = df.merge(
    macro[[c for c in MACRO_COLUMNS if c in macro.columns]],
    on="month",
    how="left",
)
print("After macro join:", df.shape)
macro_preview = [c for c in MACRO_COLUMNS if c in df.columns and c != "month"]
df[["order_date", "month", *macro_preview]].head(3)

Join keys: superstore.month = macro.month
Macro rows: 48 | Unique months in superstore: 48
After macro join: (9994, 35)


,order_date,month,cpi_index,inflation_mom,inflation_yoy,unemployment_rate,interest_rate
0,2018-11-08,2018-11,252.594,-0.0007,0.02147,3.8,2.20
1,2018-11-08,2018-11,252.594,-0.0007,0.02147,3.8,2.20
2,2018-06-12,2018-06,251.018,0.0009,0.02808,4.0,1.82


In [10]:
# --- 3) JOIN weather on date + city + state ---
weather_join = weather.copy()
weather_join["date"] = pd.to_datetime(weather_join["date"]).dt.normalize()
weather_cols = [
    "date", "city", "state", "temp_c", "rain_mm", "snow_cm", "precip_mm",
    "weather_code", "wind_gust_kmh", "had_rain", "had_snow", "had_major_storm",
]

print("Join keys: superstore.date + city + state = weather.date + city + state")
print("Weather rows:", len(weather_join))

df = df.merge(weather_join[weather_cols], on=["date", "city", "state"], how="left")
print("After weather join:", df.shape)
df[["order_date", "city", "state", "temp_c", "rain_mm", "snow_cm", "had_major_storm"]].head(3)

Join keys: superstore.date + city + state = weather.date + city + state
Weather rows: 4735
After weather join: (9994, 44)


,order_date,city,state,temp_c,rain_mm,snow_cm,had_major_storm
0,2018-11-08,Henderson,Kentucky,4.9,0.0,0.0,False
1,2018-11-08,Henderson,Kentucky,4.9,0.0,0.0,False
2,2018-06-12,Los Angeles,California,22.0,0.0,0.0,False


## Step 3 — Join synthetic loyalty CRM

**CRM (Customer Relationship Management)** is the system retailers use to track who customers are and how they engage with the loyalty program — tier status, points balance, marketing preferences, and behavioral scores. Superstore has no CRM export, so we **simulate** a one-row-per-customer lookup table and join it.

**Join:** `LEFT JOIN` on `customer_id`  
Same customer gets the same tier/points on every transaction row.

Attributes are generated with **business-informed distributions** (not independent random draws): tier drives points ranges, app engagement, discount sensitivity, and email opt-in rate — the way real loyalty programs tend to behave.

In [12]:
rng = np.random.default_rng(42)
customer_ids = df["customer_id"].drop_duplicates().sort_values()

# Tier mix: most members sit in lower tiers (typical pyramid).
TIERS = ["Bronze", "Silver", "Gold", "Platinum"]
TIER_PROBS = [0.45, 0.30, 0.20, 0.05]

# Higher tiers accumulate more points over time.
POINTS_BY_TIER = {
    "Bronze": (100, 1000),
    "Silver": (500, 3000),
    "Gold": (2000, 6000),
    "Platinum": (5000, 12000),
}

# Engaged members use the app more; Platinum averages ~0.85 vs Bronze ~0.30.
APP_USAGE_MEAN = {"Bronze": 0.30, "Silver": 0.50, "Gold": 0.70, "Platinum": 0.85}

# Discount sensitivity usually falls as tier rises (Platinum cares less about promos).
DISCOUNT_SENSITIVITY_MEAN = {"Bronze": 0.75, "Silver": 0.55, "Gold": 0.35, "Platinum": 0.20}

# Email opt-in rises with engagement; more active tiers opt in more often.
EMAIL_OPT_IN_P = {"Bronze": 0.55, "Silver": 0.68, "Gold": 0.80, "Platinum": 0.88}

crm_rows = []
for _ in customer_ids:
    tier = rng.choice(TIERS, p=TIER_PROBS)
    low, high = POINTS_BY_TIER[tier]
    crm_rows.append({
        "customer_id": _,
        "tier": tier,
        "points_balance": rng.integers(low, high),
        "email_opt_in": rng.random() < EMAIL_OPT_IN_P[tier],
        "app_usage_score": round(float(np.clip(rng.normal(APP_USAGE_MEAN[tier], 0.15), 0, 1)), 2),
        "discount_sensitivity": round(float(np.clip(rng.normal(DISCOUNT_SENSITIVITY_MEAN[tier], 0.12), 0, 1)), 2),
    })

crm = pd.DataFrame(crm_rows)

print("CRM lookup table:", crm.shape, "(one row per customer)")
crm.head(3)

CRM lookup table: (793, 6) (one row per customer)


,customer_id,tier,points_balance,email_opt_in,app_usage_score,discount_sensitivity
0,AA-10315,Gold,4618,False,0.84,0.12
1,AA-10375,Platinum,8072,True,0.80,0.20
2,AA-10480,Silver,1750,False,0.51,0.69


In [15]:
# Assign a stable unique ID to each order line (one row = one transaction).
df = df.reset_index(drop=True)
df["transaction_id"] = df.index.map(lambda i: f"txn_{i:07d}")

# Attach synthetic loyalty attributes: every line inherits its customer's CRM profile.
# LEFT JOIN keeps all Superstore rows even if a customer were missing from crm.
print("Join keys: superstore.customer_id = crm.customer_id")
fact = df.merge(crm, on="customer_id", how="left")
print("fact_transactions shape:", fact.shape)
fact[["transaction_id", "customer_id", "sales", "tier", "points_balance", "cpi_index", "temp_c"]].head(3)

Join keys: superstore.customer_id = crm.customer_id
fact_transactions shape: (9994, 50)


,transaction_id,customer_id,sales,tier,points_balance,cpi_index,temp_c
0,txn_0000000,CG-12520,261.96,Bronze,906,252.594,4.9
1,txn_0000001,CG-12520,731.94,Bronze,906,252.594,4.9
2,txn_0000002,DV-13045,14.62,Gold,3336,251.018,22.0


## Step 4 — Roll up to `dim_customers`

No external join — just **group by `customer_id`** on `fact_transactions`.

**Grain change:** line items → one row per customer

**Why:** loyalty and CRM analytics usually work at the **customer** level (not every order line). Rolling up gives one profile per member with lifetime value, recency, and CRM attributes in a single row.

**Where we'll use it:** customer segmentation / RFM, inactivity churn, tier-based targeting, and next-best-action — saved as `data/modeling/dim_customers.parquet`.

In [16]:
dim = fact.groupby("customer_id", as_index=False).agg(
    customer_name=("customer_name", "first"),
    segment=("segment", "first"),
    home_region=("region", "first"),
    first_order_date=("order_date", "min"),
    last_order_date=("order_date", "max"),
    total_orders=("order_id", "nunique"),
    total_sales=("sales", "sum"),
    total_profit=("profit", "sum"),
    avg_discount=("discount", "mean"),
    tier=("tier", "first"),
    points_balance=("points_balance", "first"),
    email_opt_in=("email_opt_in", "first"),
    app_usage_score=("app_usage_score", "first"),
    discount_sensitivity=("discount_sensitivity", "first"),
)
dim["avg_order_value"] = dim["total_sales"] / dim["total_orders"].clip(lower=1)

print("dim_customers shape:", dim.shape)
dim.head(3)

dim_customers shape: (793, 16)


,customer_id,customer_name,segment,home_region,first_order_date,last_order_date,total_orders,total_sales,total_profit,avg_discount,tier,points_balance,email_opt_in,app_usage_score,discount_sensitivity,avg_order_value
0,AA-10315,Alex Avila,Consumer,Central,2016-03-31,2019-06-29,5,5563.560,-362.8825,0.090909,Gold,4618,False,0.84,0.12,1112.712000
1,AA-10375,Allen Armold,Consumer,West,2016-04-21,2019-12-11,9,1056.390,277.3824,0.080000,Platinum,8072,True,0.80,0.20,117.376667
2,AA-10480,Andrew Allen,Consumer,South,2016-05-04,2019-04-15,4,1790.512,435.8274,0.016667,Silver,1750,False,0.51,0.69,447.628000


## Step 5 — Telco, Instacart & Online Retail (NOT joined to Superstore)

These are **separate modeling universes** with different customer IDs — never merged into `fact_transactions`.

| Source | Outputs | Use |
|--------|---------|-----|
| IBM Telco | `telco_customers.parquet` | Labeled churn classifier |
| Instacart (Kaggle) | `instacart_orders`, `instacart_baskets`, `instacart_users` | Basket / reorder models |
| UCI Online Retail I | `online_retail_transactions`, `online_retail_customers` | CLV & seasonality at scale |

In [17]:
# --- Telco: download raw CSV, add churn_flag ---
telco_path = DATA_RAW / "us_telco" / "telco_customer_churn.csv"

if not telco_path.exists():
    telco_path.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading Telco churn data...")
    r = requests.get(US_TELCO_URL, timeout=120)
    r.raise_for_status()
    telco_path.write_bytes(r.content)
else:
    print(f"Using cached file → {telco_path}")

telco = pd.read_csv(telco_path).rename(columns={"customerID": "telco_customer_id"})
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")
telco["churn_flag"] = telco["Churn"].eq("Yes").astype(int)

print("Telco shape:", telco.shape)
print("Churn rate:", telco["churn_flag"].mean().round(3))
print("Sample telco_customer_id:", telco["telco_customer_id"].iloc[0])
print("Sample superstore customer_id:", fact["customer_id"].iloc[0])
print("→ Different ID systems, no join possible")
telco[["telco_customer_id", "tenure", "MonthlyCharges", "Churn", "churn_flag"]].head(3)

Using cached file → c:\Users\ahmed\Desktop\Projects\LoyaltySim-AI\data\raw\us_telco\telco_customer_churn.csv
Telco shape: (7043, 22)
Churn rate: 0.265
Sample telco_customer_id: 7590-VHVEG
Sample superstore customer_id: CG-12520
→ Different ID systems, no join possible


,telco_customer_id,tenure,MonthlyCharges,Churn,churn_flag
0,7590-VHVEG,1,29.85,No,0
1,5575-GNVDE,34,56.95,No,0
2,3668-QPYBK,2,53.85,Yes,1


In [19]:
# --- Instacart: orders, user rollups, basket lines (same as build_datasets.py) ---
instacart_raw = DATA_RAW / "instacart"
instacart_orders = build_instacart_orders(instacart_raw)
instacart_baskets = build_instacart_baskets(instacart_raw)

if instacart_orders is not None:
    instacart = instacart_orders.groupby("user_id", as_index=False).agg(
        total_orders=("order_id", "count"),
        avg_days_since_prior=("days_since_prior_order", "mean"),
        reorder_rate=("days_since_prior_order", lambda s: s.notna().mean()),
    )
    print("Instacart orders:", instacart_orders.shape)
    print("Instacart users:", instacart.shape)
    print("Instacart baskets:", instacart_baskets.shape if instacart_baskets is not None else "skipped")
else:
    instacart = None
    instacart_baskets = None
    print("Instacart skipped — configure Kaggle CLI and re-run, or place raw CSVs in data/raw/instacart/")

instacart.head(3) if instacart is not None else None

Instacart users shape: (206209, 4)


,user_id,total_orders,avg_days_since_prior,reorder_rate
0,1,11,19.000000,0.909091
1,2,15,16.285714,0.933333
2,3,13,12.000000,0.923077


In [ ]:
# --- Online Retail I (UCI): separate CLV universe — same helper as build_datasets.py ---
try:
    online_retail_txn = load_online_retail(DATA_RAW)
    online_retail_dim = build_online_retail_customers(online_retail_txn)
    print("Online Retail transactions:", online_retail_txn.shape)
    print("Online Retail customers:", online_retail_dim.shape)
    print("Date span:", online_retail_txn["order_date"].min(), "→", online_retail_txn["order_date"].max())
except Exception as exc:
    online_retail_txn = None
    online_retail_dim = None
    print(f"Online Retail skipped: {exc}")

online_retail_dim.head(3) if online_retail_dim is not None else None

## Step 6 — Customer feature store

Built via **`scripts/feature_store.py`** (same function `build_datasets.py` calls).

Outputs RFM scores, `rfm_segment`, 16 engineered features, and `inactivity_churn_label`.

In [ ]:
customer_features = build_customer_features(fact, dim)
print("Feature store:", customer_features.shape)
customer_features[["customer_id", "rfm_segment", "RFM_score", "inactivity_churn_label"]].head()

## Step 7 — Uplift & next-best-action (synthetic)

Built via **`scripts/modeling_extensions.py`** — seeded RCT uplift campaigns + simulated NBA offer log.

In [ ]:
uplift_campaigns = build_uplift_campaigns(fact, customer_features)
nba_offer_catalog = build_nba_catalog()
nba_offer_events = build_nba_events(fact, customer_features)

print("Uplift campaigns:", uplift_campaigns.shape)
print("NBA catalog:", nba_offer_catalog.shape)
print("NBA events:", nba_offer_events.shape)
uplift_campaigns.head(3)

## Step 8 — Save modeling Parquet files

Mirrors **`build_datasets.run()`** — all outputs land in `data/modeling/`.

In [20]:
DATA_MODELING.mkdir(parents=True, exist_ok=True)

outputs = {
    "fact_transactions": DATA_MODELING / "fact_transactions.parquet",
    "dim_customers": DATA_MODELING / "dim_customers.parquet",
    "customer_features": DATA_MODELING / "customer_features.parquet",
    "telco_customers": DATA_MODELING / "telco_customers.parquet",
    "uplift_campaigns": DATA_MODELING / "uplift_campaigns.parquet",
    "nba_offer_catalog": DATA_MODELING / "nba_offer_catalog.parquet",
    "nba_offer_events": DATA_MODELING / "nba_offer_events.parquet",
}

fact.to_parquet(outputs["fact_transactions"], index=False)
dim.to_parquet(outputs["dim_customers"], index=False)
customer_features.to_parquet(outputs["customer_features"], index=False)
telco.to_parquet(outputs["telco_customers"], index=False)
uplift_campaigns.to_parquet(outputs["uplift_campaigns"], index=False)
nba_offer_catalog.to_parquet(outputs["nba_offer_catalog"], index=False)
nba_offer_events.to_parquet(outputs["nba_offer_events"], index=False)

if instacart_orders is not None:
    outputs["instacart_orders"] = DATA_MODELING / "instacart_orders.parquet"
    outputs["instacart_users"] = DATA_MODELING / "instacart_users.parquet"
    instacart_orders.to_parquet(outputs["instacart_orders"], index=False)
    instacart.to_parquet(outputs["instacart_users"], index=False)
    if instacart_baskets is not None:
        outputs["instacart_baskets"] = DATA_MODELING / "instacart_baskets.parquet"
        instacart_baskets.to_parquet(outputs["instacart_baskets"], index=False)

if online_retail_txn is not None:
    outputs["online_retail_transactions"] = DATA_MODELING / "online_retail_transactions.parquet"
    outputs["online_retail_customers"] = DATA_MODELING / "online_retail_customers.parquet"
    online_retail_txn.to_parquet(outputs["online_retail_transactions"], index=False)
    online_retail_dim.to_parquet(outputs["online_retail_customers"], index=False)

for name, path in outputs.items():
    print(f"{name:28s} {len(pd.read_parquet(path)):>12,} rows  →  {path.relative_to(PROJECT_ROOT)}")

fact_transactions       9,994 rows  →  c:\Users\ahmed\Desktop\Projects\LoyaltySim-AI\data\processed\fact_transactions.csv
dim_customers             793 rows  →  c:\Users\ahmed\Desktop\Projects\LoyaltySim-AI\data\processed\dim_customers.csv
telco_customers         7,043 rows  →  c:\Users\ahmed\Desktop\Projects\LoyaltySim-AI\data\processed\telco_customers.csv
instacart_users       206,209 rows  →  c:\Users\ahmed\Desktop\Projects\LoyaltySim-AI\data\processed\instacart_users.csv


## Join summary

```
Superstore (line items)
  + calendar features          (computed in-place)
  + macro                      LEFT JOIN on month
  + weather                    LEFT JOIN on date + city + state
  + synthetic CRM              LEFT JOIN on customer_id
  = fact_transactions

fact_transactions  →  GROUP BY customer_id  →  dim_customers
fact + dim         →  build_customer_features()  →  customer_features
fact + features    →  uplift + NBA helpers  →  uplift_campaigns, nba_offer_*

Telco CSV          → telco_customers              (standalone)
Instacart raw      → instacart_orders/baskets/users (standalone)
UCI Online Retail  → online_retail_transactions/customers (standalone)
```

**Production one-liner:** `python scripts/build_datasets.py` runs the same Steps 6–8 helpers automatically.